# ATML PA3 · Task 2 · SigLIP–Qwen VLM and visual grounding

Runs end-to-end with **Run All** / **Save Version** on Kaggle (GPU **T4 x2**, Internet ON).

1. clones the repo and installs dependencies
2. `task2.prepare`: COCO 2014 captions / instances + VQAv2 (questions, annotations, complementary
   pairs) → seeded selection of every split with isolation checks (`data/split_checks.json`) →
   only the selected COCO images are downloaded → SigLIP-base features (196 × 768, FP16, final
   LN once) cached; the vision encoder is then released. Feature caching is timed separately.
3. `task2.train` (main run, **196 visual tokens**, GPU 0): stage 1 caption alignment (2,250 updates),
   stage 2 visual instruction tuning with LoRA (1,500), stage 3 natural VQA (750); development
   evaluation before training, every 500 updates and at the stage ends; resumable checkpoints
   every 250 updates; then `task2.evaluate` on both final-test suites under four image conditions.
4. optional **49-token** run (2×2-pooled features, new connector / adapters) on GPU 1 in parallel
5. `task2.report` → **`/kaggle/working/task2_report.zip`**

If a version fails, attach its output under **Add Input** and run again: the cached features and
the newest checkpoint are reused (same data selection), finished evaluations are skipped.
If a COCO 2014 image dataset is attached, its `train2014/` / `val2014/` folders are used instead of
downloading images.

In [ ]:
# ---- settings ---------------------------------------------------------------------------
REPO_URL   = "https://github.com/alihasnaink/finetune-ground-scale.git"
BRANCH     = "main"
RUN_49     = True    # optional 49-visual-token experiment on the second GPU
SMOKE_TEST = False   # True = tiny end-to-end check, separate output folder

REPO_DIR = "/kaggle/working/repo"
if SMOKE_TEST:
    OUT_ROOT, ARTIFACTS, SCRATCH = "/kaggle/working/task2_smoke", "/kaggle/working/task2_smoke_artifacts", "/tmp/atml_task2_smoke"
    OVERRIDES = ['data.caption_train_images=64', 'data.grounding.train_pool_presence=64', 'data.grounding.train_pool_counting=64', 'data.vqa.train={"yes/no": 8, "number": 4, "other": 8}', 'train.stages=[{name: caption_alignment, updates: 20, connector_lr: 1.0e-3, lora_lr: null, mixture: {caption: 1.0}}, {name: visual_instruction, updates: 10, connector_lr: 1.0e-4, lora_lr: 2.0e-5, mixture: {presence: 0.5, counting: 0.25, caption: 0.25}}, {name: natural_vqa, updates: 10, connector_lr: 5.0e-5, lora_lr: 1.0e-5, mixture: {vqa: 0.7, presence: 0.1, counting: 0.1, caption: 0.1}}]', 'train.checkpoint_every=10', 'dev_eval.every=10', 'dev_eval.extra_updates=[20,30,40]', 'final_eval.bootstrap_resamples=200']
else:
    OUT_ROOT, ARTIFACTS, SCRATCH = "/kaggle/working/task2", "/kaggle/working/task2_artifacts", "/tmp/atml_task2"
    OVERRIDES = []
ZIP_PATH = f"/kaggle/working/task2{'_smoke' if SMOKE_TEST else ''}_report.zip"
CONFIGS = {"vlm196": "task2/configs/vlm196.yaml", "vlm49": "task2/configs/vlm49.yaml"}

In [ ]:
!nvidia-smi
import os, subprocess, sys, time, glob
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], text=True))

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-task2.txt
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, "enable a GPU accelerator (T4 x2 recommended)"
# an attached COCO 2014 image dataset (folders train2014/ and val2014/) avoids downloading images
roots = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train2014", recursive=True)
                if os.path.isdir(p) and os.path.isdir(os.path.join(os.path.dirname(p), "val2014"))})
if roots:
    OVERRIDES = OVERRIDES + [f"paths.coco_image_roots={roots!r}"]
print("COCO image roots:", roots or "none (images are downloaded)")

In [ ]:
ENV = os.environ | {"T2_OUT_ROOT": OUT_ROOT, "T2_SCRATCH_ROOT": SCRATCH, "T2_ARTIFACTS_ROOT": ARTIFACTS,
                    "T2_PRIOR_SEARCH": "/kaggle/input", "PYTHONUNBUFFERED": "1",
                    "TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                    "HF_HUB_DISABLE_PROGRESS_BARS": "1"}

def _cmd(module, args, config):
    cmd = [sys.executable, "-m", f"task2.{module}", "--config", config, *args]
    return cmd + (["--set", *OVERRIDES] if OVERRIDES else [])

def run(module, *args, gpu="0", config=CONFIGS["vlm196"]):
    t0 = time.time()
    p = subprocess.Popen(_cmd(module, args, config), cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"task2.{module} failed (exit {p.returncode})")
    print(f"--- task2.{module} finished in {(time.time() - t0) / 60:.1f} min")

def run_lane_bg(steps, gpu, logfile):
    """steps: [(module, args, config)] run one after another in the background on `gpu`."""
    import shlex
    script = "set -e\n" + "\n".join(shlex.join(_cmd(m, a, c)) for m, a, c in steps)
    return subprocess.Popen(["bash", "-c", script], cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu},
                            stdout=open(logfile, "w"), stderr=subprocess.STDOUT, text=True)

os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# 1) annotations, selection (+ checks), images, SigLIP feature cache (reused if attached)
run("prepare")
print(open(f"{OUT_ROOT}/data/split_checks.json").read()[:3000])

In [ ]:
# 2) optional 49-token run on GPU 1 in the background (train -> final evaluation)
lane = None
if RUN_49 and N_GPU >= 2:
    LANE_LOG = f"{OUT_ROOT}/lane_vlm49_gpu1.log"
    lane = run_lane_bg([("train", [], CONFIGS["vlm49"]), ("evaluate", [], CONFIGS["vlm49"])], "1", LANE_LOG)
    print("vlm49 lane started on GPU 1 ->", LANE_LOG)

In [ ]:
# 3) main run (196 visual tokens) on GPU 0: three-stage training, then final evaluation
run("train", gpu="0")
run("evaluate", gpu="0")

In [ ]:
# 4) optional run: wait for GPU 1 (or run it here on a single GPU)
if lane is not None:
    while lane.poll() is None:
        time.sleep(30)
    print(open(LANE_LOG).read()[-6000:])
    assert lane.returncode == 0, f"vlm49 lane failed; see {LANE_LOG}"
elif RUN_49:
    run("train", gpu="0", config=CONFIGS["vlm49"])
    run("evaluate", gpu="0", config=CONFIGS["vlm49"])

In [ ]:
# 5) tables, figures, qualitative pairs and the downloadable zip
run("report", "--zip", ZIP_PATH)
from pathlib import Path
from IPython.display import Markdown, display, Image, FileLink
rep = Path(OUT_ROOT) / "report"
for t in ["data_sizes", "final_results", "image_advantages", "swap_analysis", "resources"]:
    f = rep / "tables" / f"{t}.md"
    if f.exists():
        display(Markdown(f.read_text()))
for f in sorted((rep / "figures").glob("*.png")):
    display(Image(filename=str(f)))
display(FileLink(os.path.relpath(ZIP_PATH, "/kaggle/working")))

In [ ]:
!du -sh {OUT_ROOT} {ARTIFACTS} {ZIP_PATH} {SCRATCH} 2>/dev/null

In [ ]:
import json
for run in ["vlm196", "vlm49"]:
    rows = [json.loads(l) for l in open(f"{OUT_ROOT}/runs/{run}/final_eval/samples.jsonl")]
    for cond in ["correct", "no_image"]:
        print(run, cond)
        for r in [r for r in rows if r["condition"] == cond][:6]:
            print("   Q:", r["question"], "| ref:", r["answer"], "| pred:", repr(r["prediction"]), "| tokens", r["n_tokens"])
